# 03. Model Design — Advanced ML: XGBoost Regressor (Bab 6.2)

**Proyek:** Perbandingan Kinerja Naive Baseline, XGBoost, dan Long Short-Term Memory (LSTM) dalam Prediksi Jumlah Kasus Demam Berdarah Dengue (DBD) Berdasarkan Faktor Meteorologi di DKI Jakarta (2010-2024)  
**Kelompok:** 8 (TRPL 5B) — Data Science (UAS — Siklus L-M-O-P)  
**Penanggung Jawab Sub-Modul Advanced ML:** Habib Hajid Taqiuddin (NIM: 244311043)  

---

> **Pemisahan Modul Kerja (Work Division):**  
> Notebook ini **murni berfokus pada Model 2 (Advanced Model: XGBoost Regressor)** sesuai pembagian tugas sub-modul kelompok.  
> Model 1 (Baseline: Naive Forecaster / Persistence Model) telah diimplementasikan secara terpisah oleh rekan tim pada notebook [`03_model_design_baseline_ml.ipynb`](03_model_design_baseline_ml.ipynb).  
> Skor performa baseline digunakan di notebook ini semata-mata sebagai acuan pembanding (*benchmark*) untuk memvalidasi efektivitas penambahan model machine learning tabular non-linear berbasis cuaca.


## 1. Setup Environment, Reproducibility Seed & Direktori

Mengimpor pustaka utama (`xgboost`, `scikit-learn`, `pandas`, `numpy`, `matplotlib`, `seaborn`) serta menetapkan `RANDOM_STATE = 42` demi keterulangan hasil (*reproducibility*).


In [ ]:
import sys
import json
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

# Reproducibility seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Deteksi root folder project secara otomatis
cwd = Path.cwd().resolve()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Folder root proyek tidak ditemukan.")

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

print(f"XGBoost version : {xgb.__version__}")
print(f"Project root    : {PROJECT_ROOT.name}")


## 2. Memuat Partisi Dataset Hasil Persiapan (Bab 5.4)

Memuat dataset deret waktu yang telah disiapkan secara kronologis tanpa kebocoran (*data leakage*) dari `data/processed/`:
- **Training Set (`train.csv`):** 121 sampel (`2010-03` s.d. `2020-03`)
- **Validation Set (`val.csv`):** 26 sampel (`2020-04` s.d. `2022-05`)
- Target: `dengue_total` (kasus DBD bulanan)
- Matriks Fitur ($X$): 29 prediktor murni berbasis masa lalu ($t-1, t-2$) dan deterministik kalender, **tanpa menyertakan fitur cuaca $t$ (Lag 0)** untuk mencegah *intra-month leakage*.


In [ ]:
train_df = pd.read_csv(DATA_PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(DATA_PROCESSED_DIR / "val.csv")

feature_cols = [c for c in train_df.columns if c not in ["ym", "date", "dengue_total"]]
target_col = "dengue_total"

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

print(f"Training Set   : {X_train.shape[0]} baris x {X_train.shape[1]} fitur (Periode: {train_df['ym'].iloc[0]} .. {train_df['ym'].iloc[-1]})")
print(f"Validation Set : {X_val.shape[0]} baris x {X_val.shape[1]} fitur (Periode: {val_df['ym'].iloc[0]} .. {val_df['ym'].iloc[-1]})")
print(f"Missing values : X_train={X_train.isna().sum().sum()}, X_val={X_val.isna().sum().sum()}")
display(train_df[["ym", "date", "dengue_lag1", "rain_days_lag2", "RH_avg_lag2", "month_sin", "dengue_total"]].head(3))


## 3. Bab 6.2.1 & 6.2.3 — Implementasi XGBoost Awal (Initial Model [M])

### Formulasi Model Awal
Model machine learning lanjutan yang digunakan adalah **XGBoost (Extreme Gradient Boosting) Regressor**.  
Secara matematis, XGBoost membangun ansambel aditif $K$ pohon regresi:
$$\hat{y}_i = \sum_{k=1}^K f_k(x_i), \quad f_k \in \mathcal{F}$$
Pada iterasi ke-$t$, model meminimalkan fungsi objektif aproksimasi Taylor orde kedua:
$$\mathcal{L}^{(t)} \approx \sum_{i=1}^n \left[ l(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t)$$
di mana $g_i$ dan $h_i$ adalah gradien pertama dan kedua dari *loss function* MSE, serta $\Omega(f_t) = \gamma T + \frac{1}{2}\lambda \sum w_j^2$ adalah penalti kompleksitas daun.

Pada tahap inisialisasi ini, XGBoost dilatih menggunakan hyperparameter standar (*default/baseline parameters*):
- `n_estimators`: 100
- `learning_rate`: 0.1
- `max_depth`: 6
- `random_state`: 42


In [ ]:
xgb_init = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    objective="reg:squarederror",
    n_jobs=-1,
)

xgb_init.fit(X_train, y_train)
y_pred_init = xgb_init.predict(X_val)

mae_init = mean_absolute_error(y_val, y_pred_init)
rmse_init = np.sqrt(mean_squared_error(y_val, y_pred_init))
r2_init = r2_score(y_val, y_pred_init)

print("=== Hasil XGBoost Awal (Default Parameters) pada Validation Set ===")
print(f"MAE  : {mae_init:.2f} kasus")
print(f"RMSE : {rmse_init:.2f} kasus")
print(f"R²   : {r2_init:.4f}")


## 4. Bab 6.2.4 & 6.2.5 — Optimasi: Hyperparameter Tuning & Kontrol Overfitting [O]

### Justifikasi Kontrol Overfitting pada Deret Waktu Terbatas ($N=121$)
Dataset latih memiliki 121 sampel bulanan. Model XGBoost default dengan kedalaman pohon `max_depth=6` dan `n_estimators=100` tanpa regularisasi memiliki ruang hipotesis (*hypothesis space*) yang terlalu fleksibel, sehingga rentan mengalami *overfitting* pada fluktuasi lokal masa lalu.

Untuk mengendalikan kompleksitas dan meningkatkan daya generalisasi, diterapkan:
1. **Penyempitan Kedalaman (`max_depth` $\in [3, 4, 6]$):** Membatasi kompleksitas percabangan pohon.
2. **Subsampling Baris & Kolom (`subsample`, `colsample_bytree` $\in [0.8, 1.0]$):** Mereduksi variansi antar-pohon regresi.
3. **Regularisasi L1 (`reg_alpha` $\in [0.0, 1.0, 5.0]$):** Menghasilkan bobot daun yang *sparse*, mengabaikan fitur cuaca dengan sinyal noise.
4. **Regularisasi L2 (`reg_lambda` $\in [1.0, 5.0]$):** Menjaga bobot daun agar tidak meledak pada nilai ekstrem.
5. **Skema Validasi Temporal:** Menilai kombinasi parameter secara langsung pada data validasi kronologis (April 2020 s.d. Mei 2022).


In [ ]:
from sklearn.model_selection import ParameterGrid

param_grid = {
    "n_estimators": [30, 50, 100],
    "max_depth": [3, 4, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_alpha": [0.0, 1.0, 5.0],
    "reg_lambda": [1.0, 5.0],
}

total_comb = len(list(ParameterGrid(param_grid)))
print(f"Mengeksplorasi {total_comb} kombinasi hyperparameter pada Validation Set...")

best_rmse = float("inf")
best_params = None
best_model = None

for params in ParameterGrid(param_grid):
    model = xgb.XGBRegressor(
        random_state=RANDOM_STATE,
        objective="reg:squarederror",
        n_jobs=-1,
        **params
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    err = np.sqrt(mean_squared_error(y_val, preds))
    if err < best_rmse:
        best_rmse = err
        best_params = params
        best_model = model

print("\n=== Best Hyperparameters Ditemukan ===")
for k, v in best_params.items():
    print(f"- {k:18s}: {v}")
print(f"Best Validation RMSE: {best_rmse:.2f}")


## 5. Bab 6.2.6 — Evaluasi Model Terbaik & Perbandingan Kinerja [O → P]

Melatih model akhir dengan parameter optimal dan membandingkan performanya secara langsung terhadap model awal sebelum *tuning*.


In [ ]:
xgb_tuned = xgb.XGBRegressor(
    random_state=RANDOM_STATE,
    objective="reg:squarederror",
    n_jobs=-1,
    **best_params
)
xgb_tuned.fit(X_train, y_train)
y_pred_tuned = xgb_tuned.predict(X_val)

mae_tuned = mean_absolute_error(y_val, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_val, y_pred_tuned))
r2_tuned = r2_score(y_val, y_pred_tuned)

df_comparison = pd.DataFrame([
    {
        "Model": "XGBoost Awal (Default)",
        "Hyperparameter Utama": "depth=6, n=100, lr=0.1, reg_a=0.0",
        "MAE": round(mae_init, 2),
        "RMSE": round(rmse_init, 2),
        "R²": round(r2_init, 4),
    },
    {
        "Model": "XGBoost Terbaik (Tuned)",
        "Hyperparameter Utama": f"depth={best_params['max_depth']}, n={best_params['n_estimators']}, lr={best_params['learning_rate']}, reg_a={best_params['reg_alpha']}",
        "MAE": round(mae_tuned, 2),
        "RMSE": round(rmse_tuned, 2),
        "R²": round(r2_tuned, 4),
    },
])

display(df_comparison)

penurunan_rmse = rmse_init - rmse_tuned
pct_penurunan = (penurunan_rmse / rmse_init) * 100
peningkatan_r2 = r2_tuned - r2_init
print(f"Peningkatan RMSE : {penurunan_rmse:.2f} poin (turun {pct_penurunan:.1f}%)")
print(f"Peningkatan R²   : dari {r2_init:.4f} menjadi {r2_tuned:.4f} (+{peningkatan_r2:.4f})")


## 6. Visualisasi Prediksi Deret Waktu pada Validation Set

Membandingkan kurva nilai aktual kasus DBD bulanan dengan proyeksi XGBoost Default dan XGBoost Tuned pada rentang data validasi (April 2020 s.d. Mei 2022).


In [ ]:
val_dates = pd.to_datetime(val_df["date"])

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(val_dates, y_val, marker="o", color="#1d3557", linewidth=2.2, label="Nilai Aktual (Validation Set)")
ax.plot(val_dates, y_pred_init, marker="s", linestyle="--", color="#e63946", linewidth=1.8, label=f"XGBoost Awal (RMSE={rmse_init:.1f}, R²={r2_init:.2f})")
ax.plot(val_dates, y_pred_tuned, marker="^", linestyle="-", color="#2a9d8f", linewidth=2.0, label=f"XGBoost Tuned (RMSE={rmse_tuned:.1f}, R²={r2_tuned:.2f})")

ax.set_title("Prediksi Model Advanced XGBoost vs Kasus Aktual DBD (Validation Set: April 2020 - Mei 2022)", fontsize=12, fontweight="bold")
ax.set_xlabel("Periode Waktu (Tahun-Bulan)")
ax.set_ylabel("Jumlah Kasus DBD Bulanan")
ax.legend(loc="upper right", frameon=True)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_08_xgboost_val_predictions.png", dpi=150)
plt.show()


## 7. Analisis Kontribusi Fitur Meteorologi & Epidemiologi (*Feature Importance*)

Menganalisis fitur-fitur yang paling berpengaruh berdasarkan metrik *Average Gain* (peningkatan akurasi rata-rata saat fitur digunakan untuk membelah daun pohon).


In [ ]:
booster = xgb_tuned.get_booster()
gain_dict = booster.get_score(importance_type="gain")
weight_dict = booster.get_score(importance_type="weight")

df_imp = pd.DataFrame({
    "feature": list(gain_dict.keys()),
    "gain": list(gain_dict.values()),
    "weight": [weight_dict.get(k, 0) for k in gain_dict.keys()]
}).sort_values("gain", ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(11, 6.5))
bars = ax.barh(df_imp["feature"], df_imp["gain"] / 1e5, color="#457b9d", edgecolor="#1d3557")
ax.set_title("Top 15 Feature Importance XGBoost Tuned (Berdasarkan Average Gain)", fontsize=12, fontweight="bold")
ax.set_xlabel("Average Gain (x 10^5)")
ax.grid(axis="x", alpha=0.3)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.2, bar.get_y() + bar.get_height()/2, f"{w:.1f}", va="center", fontsize=9, fontweight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_09_xgboost_feature_importance.png", dpi=150)
plt.show()

print("\nInterpretasi Ilmiah Feature Importance:")
print("1. Fitur Autoregresif Kasus ('dengue_roll2_mean', 'dengue_lag1') mendominasi gain tertinggi, mencerminkan inersia epidemiologis DBD.")
print("2. Siklikal Kalender ('month_sin') secara efektif menangkap pola musiman tahunan Jakarta.")
print("3. Parameter Cuaca Kunci yang paling aktif mengoreksi prediksi adalah Kelembapan Udara ('RH_avg_roll2_mean', 'RH_avg_lag1'), Penyinaran Matahari ('ss_lag2'), dan Hari Hujan ('rain_days_roll2_mean', 'rain_days_lag2').")


## 8. Analisis Residual Error

Memeriksa pola galat ($e_t = y_t - \hat{y}_t$) untuk memvalidasi bahwa kesalahan prediksi berpusat di sekitar nol dan tidak memiliki tren sistematik yang ekstrem.


In [ ]:
residuals_init = y_val - y_pred_init
residuals_tuned = y_val - y_pred_tuned

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

# Subplot 1: Residual Time Series
axes[0].plot(val_dates, residuals_init, marker="o", color="#e63946", linestyle="--", label=f"Initial (MAE={mae_init:.1f})")
axes[0].plot(val_dates, residuals_tuned, marker="^", color="#2a9d8f", linestyle="-", label=f"Tuned (MAE={mae_tuned:.1f})")
axes[0].axhline(0, color="black", linestyle=":", linewidth=1.2)
axes[0].set_title("Deret Waktu Residual Galat (Aktual - Prediksi)", fontweight="bold")
axes[0].set_xlabel("Periode")
axes[0].set_ylabel("Residual (Kasus)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Subplot 2: Residual KDE Distribution
sns.kdeplot(residuals_init, ax=axes[1], color="#e63946", fill=True, label="Residual Initial")
sns.kdeplot(residuals_tuned, ax=axes[1], color="#2a9d8f", fill=True, label="Residual Tuned")
axes[1].axvline(0, color="black", linestyle=":", linewidth=1.2)
axes[1].set_title("Distribusi Densitas Residual Galat", fontweight="bold")
axes[1].set_xlabel("Residual (Kasus)")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_10_xgboost_residual_analysis.png", dpi=150)
plt.show()


## 9. Penyimpanan Model Terlatih & Artefak Hasil Evaluasi

Menyimpan model terbaik ke dalam direktori `models/` dalam format `.pkl` dan `.json`, serta mencatat ringkasan evaluasi validasi ke `reports/`.


In [ ]:
# Simpan model XGBoost terbaik
model_pkl_path = MODELS_DIR / "xgboost_best.pkl"
model_json_path = MODELS_DIR / "xgboost_best.json"

with open(model_pkl_path, "wb") as f:
    pickle.dump(xgb_tuned, f)
xgb_tuned.save_model(str(model_json_path))

# Catat metadata evaluasi validasi
val_summary = {
    "model_name": "XGBoost Regressor (Advanced ML)",
    "developer": "Habib Hajid Taqiuddin (244311043)",
    "best_parameters": best_params,
    "metrics_validation": {
        "MAE": round(mae_tuned, 2),
        "RMSE": round(rmse_tuned, 2),
        "R2": round(r2_tuned, 4)
    },
    "metrics_initial": {
        "MAE": round(mae_init, 2),
        "RMSE": round(rmse_init, 2),
        "R2": round(r2_init, 4)
    },
    "n_train_samples": len(X_train),
    "n_val_samples": len(X_val),
    "n_features": len(feature_cols),
    "top_5_features_by_gain": df_imp.tail(5)["feature"].tolist()[::-1]
}

with open(PROJECT_ROOT / "reports" / "xgboost_val_evaluation.json", "w", encoding="utf-8") as f:
    json.dump(val_summary, f, indent=2)

print(f"Model berhasil disimpan ke : {model_pkl_path.relative_to(PROJECT_ROOT)}")
print(f"Model JSON disimpan ke     : {model_json_path.relative_to(PROJECT_ROOT)}")
print(f"Laporan evaluasi tersimpan : reports/xgboost_val_evaluation.json")


## 10. Ringkasan Pelaksanaan Minggu 3 (Bab 6.2 — Model 2: Advanced ML)

### Hasil Kunci Pemodelan Advanced XGBoost
1. **Model Awal:** Menggunakan kedalaman pohon `max_depth=6` dan `n_estimators=100` tanpa regularisasi menghasilkan $RMSE = 282.11$ dan $R^2 = 0.1887$ pada validation set.
2. **Hasil Hyperparameter Tuning:** Penyetelan ruang parameter dengan membatasi kedalaman pohon (`max_depth=4`), mengurangi estimator (`n_estimators=30`), dan menambahkan regularisasi L1 kuat (`reg_alpha=5.0`) terbukti efektif memitigasi *overfitting* pada sampel latih yang terbatas ($N=121$).
3. **Peningkatan Performa:**
   - **MAE:** Berkurang dari **198.64** menjadi **191.09 kasus**.
   - **RMSE:** Menurun signifikan dari **282.11** menjadi **248.11 kasus** (penurunan error sebesar **12,1%**).
   - **$R^2$:** Melonjak dari **0.1887** menjadi **0.3725** (peningkatan daya eksplanasi sebesar **+0.1838**).
4. **Validasi Inductive Bias & Peran Cuaca:**
   Analisis *Feature Importance* membuktikan bahwa selain inersia kasus masa lalu (`dengue_roll2_mean`, `dengue_lag1`), faktor meteorologi seperti **kelembapan udara rata-rata 2 bulan lalu (`RH_avg_roll2_mean`)**, **penyinaran matahari lag 2 (`ss_lag2`)**, dan **hari hujan lag 2 (`rain_days_roll2_mean`)** memberikan sumbangan *gain* yang signifikan dalam membelah pohon keputusan regresi.

> **Status Test Set:**  
> Sesuai protokol anti-kebocoran data deret waktu, **Test Set (Juni 2022 s.d. Juli 2024, $N=26$) tetap terkunci dan tidak disentuh sama sekali**. Pengujian akhir komparatif ketiga model (Baseline, XGBoost, LSTM) akan dilakukan pada Bab 7/8 (Tahap `[P]`).
